# Preprocessing — HPP sévère

| Décision | Choix | Pourquoi |
|----------|-------|----------|
| NA | **`dropna`** | Meilleur recall/F1 qu’une imputation simple (expériences `_archive/`) |
| Split | Stratifié 80/20 | Préserve le déséquilibre |
| Encodage | `ColumnTransformer` | Quant / binaire / nominal / ordinal |

Données : `load_hpp()` → processed si dispo.

## 1. Chargement

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

sys.path.insert(0, str(Path.cwd()))
from hpp_data import TARGET, feature_lists_present, load_hpp

df, source = load_hpp()
print(f"Source : {source} — {df.shape}")
print("NA % (top) :")
print((df.isna().mean() * 100).sort_values(ascending=False).head(8).round(2))

## 2. dropna (stratégie retenue)

Implication déploiement : les nouvelles admissions doivent être **complètes**.

In [ ]:
n_before = len(df)
df_complete = df.dropna()
summary = pd.DataFrame({
    "stratégie": [f"brut ({source})", "après dropna"],
    "n_lignes": [n_before, len(df_complete)],
    "n_positifs": [
        int(df[TARGET].sum()) if TARGET in df.columns else None,
        int(df_complete[TARGET].sum()) if TARGET in df_complete.columns else None,
    ],
})
summary

## 3. Preprocessor sklearn

In [ ]:
def build_preprocessor(quant, binary, nominal, ordinal):
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    nominal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])
    ordinal_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])
    return ColumnTransformer([
        ("num", numeric, quant),
        ("bin", "passthrough", binary),
        ("nom", nominal_pipe, nominal),
        ("ord", ordinal_pipe, ordinal),
    ])


lists = feature_lists_present(df_complete)
preprocessor = build_preprocessor(
    lists["quant"], lists["binary"], lists["nominal"], lists["ordinal"]
)
print(preprocessor)
feat_cols = lists["quant"] + lists["binary"] + lists["nominal"] + lists["ordinal"]
print("n_features :", len(feat_cols))

## 4. Split stratifié

Nécessite ≥ 2 classes et assez de lignes (base processed).

In [ ]:
can_split = (
    TARGET in df_complete.columns
    and df_complete[TARGET].nunique() >= 2
    and len(df_complete) >= 50
)

if can_split:
    X = df_complete[feat_cols]
    y = df_complete[TARGET].astype(int)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    preprocessor.fit(X_train)
    print(f"Train {X_train.shape} | Test {X_test.shape}")
    print("Positifs train %:", round(100 * y_train.mean(), 2))
else:
    print(
        "Split non réalisable (extrait démo ou mono-classe).\n"
        "→ Lancez 00_Prepare_Data.ipynb avec le .xls Drive."
    )

## 5. Suite

`03_Model_Comparison.ipynb` — LogReg / RF / XGB + tracking MLflow.